In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
url = "https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv"
df = pd.read_csv(url)
df.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet
0,0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [ ]:
# Keep only what we need
df = df[['tweet', 'class']]

# Function to clean text
def clean_text(text):
    text = text.lower()                          # lowercase everything
    text = re.sub(r'rt\s+', '', text)             # remove "RT" (retweet marker)
    text = re.sub(r'@\w+', '', text)              # remove @mentions
    text = re.sub(r'http\S+|www\S+', '', text)    # remove links
    text = re.sub(r'[^a-z\s]', '', text)          # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()      # remove extra spaces
    return text

df['clean_tweet'] = df['tweet'].apply(clean_text)

df[['tweet', 'clean_tweet', 'class']].head()

,tweet,clean_tweet,class
0,!!! RT @mayasolovely: As a woman you shouldn't...,as a woman you shouldnt complain about cleanin...,2
1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...,boy dats coldtyga dwn bad for cuffin dat hoe i...,1
2,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...,dawg you ever fuck a bitch and she stato cry y...,1
3,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...,she look like a tranny,1
4,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...,the shit you hear about me might be true or it...,1


In [ ]:
# Remove stopwords (common words like "the", "is", "and" that don't carry much meaning)
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

df['final_tweet'] = df['clean_tweet'].apply(remove_stopwords)

# Convert text into numbers using TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['final_tweet']).toarray()
y = df['class']

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (24783, 5000)
Shape of y: (24783,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 19826
Testing samples: 4957


In [ ]:
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

print("Model training complete! ✅")

Model training complete! ✅


In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.891466612870688

Classification Report:
              precision    recall  f1-score   support

           0       0.52      0.16      0.24       290
           1       0.91      0.96      0.94      3832
           2       0.83      0.82      0.83       835

    accuracy                           0.89      4957
   macro avg       0.75      0.65      0.67      4957
weighted avg       0.88      0.89      0.88      4957


Confusion Matrix:
[[  46  210   34]
 [  42 3685  105]
 [   1  146  688]]


In [ ]:
def predict_toxicity(text):
    cleaned = clean_text(text)
    cleaned = remove_stopwords(cleaned)
    vectorized = tfidf.transform([cleaned]).toarray()
    prediction = model.predict(vectorized)[0]

    labels = {0: "Hate Speech 🚫", 1: "Offensive Language ⚠️", 2: "Neither / Normal ✅"}
    return labels[prediction]

# Try it out!
print(predict_toxicity("You look beautiful"))
print(predict_toxicity("I hate you so much"))
print(predict_toxicity("Have a great day!"))

Neither / Normal ✅
Offensive Language ⚠️
Neither / Normal ✅


In [ ]:
!pip install transformers torch -q

import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split as tts

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
# Take a smaller sample so training is faster
df_sample = df.sample(5000, random_state=42).reset_index(drop=True)

X_train_b, X_test_b, y_train_b, y_test_b = tts(
    df_sample['tweet'], df_sample['class'], test_size=0.2, random_state=42
)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_data(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

train_encodings = tokenize_data(X_train_b)
test_encodings = tokenize_data(X_test_b)

print("Tokenization done ✅")
print("Sample tokens:", train_encodings['input_ids'][0])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenization done ✅
Sample tokens: tensor([  101, 19387,  1030, 28486,  4213,  7317,  4059,  1024,  2006,  2275,
         1997,  1001,  2630, 26682,  2015,  1012,  5327,  2045,  2024,  2053,
         5055, 17335,  2682,  2033,  1012,  1012,  1012,  8299,  1024,  1013,
         1013,  1056,  1012,  2522,  1013,  1051, 16425,  4160, 25746, 14702,
         2571,   102,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0])


In [ ]:
class TweetDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = TweetDataset(train_encodings, y_train_b)
test_dataset = TweetDataset(test_encodings, y_test_b)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

print("DataLoaders ready ✅")

DataLoaders ready ✅


In [ ]:
model_bert = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)
model_bert.to(device)

optimizer = AdamW(model_bert.parameters(), lr=2e-5)

print("BERT model loaded ✅")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT model loaded ✅


In [ ]:
epochs = 2

model_bert.train()

for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model_bert(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} - Average Loss: {avg_loss:.4f}")

print("BERT training complete! ✅")

Epoch 1/2 - Average Loss: 0.4649
Epoch 2/2 - Average Loss: 0.2638
BERT training complete! ✅


In [ ]:
model_bert.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model_bert(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy_bert = accuracy_score(all_labels, all_preds)
print("BERT Accuracy:", accuracy_bert)

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

BERT Accuracy: 0.898

Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.26      0.34        53
           1       0.95      0.93      0.94       779
           2       0.78      0.94      0.85       168

    accuracy                           0.90      1000
   macro avg       0.74      0.71      0.71      1000
weighted avg       0.89      0.90      0.89      1000



In [ ]:
def predict_toxicity_bert(text):
    model_bert.eval()
    cleaned = clean_text(text)
    encoding = tokenizer(cleaned, truncation=True, padding=True, max_length=64, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model_bert(**encoding)
        pred = torch.argmax(output.logits, dim=1).item()

    labels = {0: "Hate Speech ", 1: "Offensive Language ", 2: "Neither / Normal "}
    return labels[pred]

print(predict_toxicity_bert("You look beautiful"))
print(predict_toxicity_bert("I hate you so much"))
print(predict_toxicity_bert("Have a great day!"))

Neither / Normal 
Offensive Language 
Neither / Normal 


In [ ]:
# Should be Normal/Neither
print(predict_toxicity_bert("You look beautiful"))
print(predict_toxicity_bert("Have a nice day"))
print(predict_toxicity_bert("I love this movie"))

# Should be Offensive/Hate
print(predict_toxicity_bert("You are so stupid"))
print(predict_toxicity_bert("I hate all of them"))
print(predict_toxicity_bert("Get lost, idiot"))

Neither / Normal 
Neither / Normal 
Neither / Normal 
Offensive Language 
Offensive Language 
Offensive Language 


In [ ]:
test_cases = [
    ("You look beautiful", "Neither / Normal "),
    ("Have a nice day", "Neither / Normal "),
    ("I love this movie", "Neither / Normal "),
    ("Thank you so much", "Neither / Normal "),
    ("You are so stupid", "Offensive Language "),
    ("Get lost, idiot", "Offensive Language "),
    ("Shut up, nobody asked", "Offensive Language "),
]

print(f"{'Sentence':<35}{'Expected':<25}{'Predicted'}")
print("-" * 90)

correct = 0
for sentence, expected in test_cases:
    predicted = predict_toxicity_bert(sentence)
    match = "OK" if expected.split()[0] in predicted else "X"
    if match == "OK":
        correct += 1
    print(f"{sentence:<35}{expected:<25}{predicted}  {match}")

print("-" * 90)
score = correct / len(test_cases) * 100
print("Model got", correct, "out of", len(test_cases), "obvious cases right (", round(score), "%)")

if score >= 70:
    print("Model is working correctly!")
else:
    print("Model may need more training or a bigger dataset.")

Sentence                           Expected                 Predicted
------------------------------------------------------------------------------------------
You look beautiful                 Neither / Normal         Neither / Normal   OK
Have a nice day                    Neither / Normal         Neither / Normal   OK
I love this movie                  Neither / Normal         Neither / Normal   OK
Thank you so much                  Neither / Normal         Neither / Normal   OK
You are so stupid                  Offensive Language       Offensive Language   OK
Get lost, idiot                    Offensive Language       Offensive Language   OK
Shut up, nobody asked              Offensive Language       Offensive Language   OK
------------------------------------------------------------------------------------------
Model got 7 out of 7 obvious cases right ( 100 %)
Model is working correctly!
